In [15]:
import sqlite3
import pandas as pd


DB_PATH = "../data/db.sqlite"
conn = sqlite3.connect(DB_PATH)

# Load all tables
company = pd.read_sql("SELECT * FROM Company", conn)
product = pd.read_sql("SELECT * FROM Product", conn)
bom = pd.read_sql("SELECT * FROM BOM", conn)
bom_component = pd.read_sql("SELECT * FROM BOM_Component", conn)
supplier = pd.read_sql("SELECT * FROM Supplier", conn)
supplier_product = pd.read_sql("SELECT * FROM Supplier_Product", conn)


In [16]:
fg_product = product.rename(
    columns={
        "Id": "FG_Id",
        "SKU": "FG_SKU",
        "Type": "FG_Type",
        "CompanyId": "CompanyId",
    }
)
rm_product = product.rename(
    columns={
        "Id": "RM_Id",
        "SKU": "RM_SKU",
        "Type": "RM_Type",
        "CompanyId": "RM_CompanyId",
    }
)
supplier_info = (
    supplier_product[["SupplierId", "ProductId"]]
    .merge(
        supplier.rename(columns={"Id": "SupplierId", "Name": "SupplierName"}),
        on="SupplierId",
    )
    .rename(columns={"ProductId": "RM_Id"})
)

df_all = (
    bom_component.rename(columns={"BOMId": "FG_Id", "ConsumedProductId": "RM_Id"})
    .merge(bom, left_on="FG_Id", right_on="Id")  # + ProducedProductId
    .merge(fg_product, on="FG_Id")  # + FG_SKU, FG_Type, CompanyId
    .merge(
        company.rename(columns={"Id": "CompanyId", "Name": "CompanyName"}),
        on="CompanyId",
    )  # + CompanyName
    .merge(rm_product, on="RM_Id")  # + RM_SKU, RM_Type
    .merge(supplier_info, on="RM_Id")[  # + SupplierId, SupplierName
        [
            "FG_Id",
            "FG_SKU",
            "FG_Type",
            "CompanyId",
            "CompanyName",
            "RM_Id",
            "RM_SKU",
            "RM_Type",
            "SupplierId",
            "SupplierName",
        ]
    ]
)

df_all


,FG_Id,FG_SKU,FG_Type,CompanyId,CompanyName,RM_Id,RM_SKU,RM_Type,SupplierId,SupplierName
0,1,FG-iherb-10421,finished-good,28,NOW Foods,506,RM-C28-glycerin-85e43afb,raw-material,18,Ingredion
1,1,FG-iherb-10421,finished-good,28,NOW Foods,509,RM-C28-safflower-oil-a84bc3ce,raw-material,1,ADM
2,1,FG-iherb-10421,finished-good,28,NOW Foods,509,RM-C28-safflower-oil-a84bc3ce,raw-material,10,Cargill
3,1,FG-iherb-10421,finished-good,28,NOW Foods,511,RM-C28-softgel-capsule-bovine-gelatin-5a1a1582,raw-material,9,Capsuline
4,1,FG-iherb-10421,finished-good,28,NOW Foods,511,RM-C28-softgel-capsule-bovine-gelatin-5a1a1582,raw-material,13,Darling Ingredients / Rousselot
...,...,...,...,...,...,...,...,...,...,...
2855,149,FG-sams-club-p03012212,finished-good,30,Nature Made,549,RM-C30-vegetarian-capsule-ea92a1d1,raw-material,11,Colorcon
2856,149,FG-sams-club-p03012212,finished-good,30,Nature Made,551,RM-C30-vitamin-c-5399011f,raw-material,27,Prinova USA
2857,149,FG-sams-club-p03012212,finished-good,30,Nature Made,551,RM-C30-vitamin-c-5399011f,raw-material,28,PureBulk
2858,149,FG-sams-club-p03012212,finished-good,30,Nature Made,552,RM-C30-vitamin-d3-e0800226,raw-material,27,Prinova USA


In [17]:
df_all.to_csv("../data/all_data.csv", index=False)


In [18]:
import pandas as pd

df = pd.read_csv("../data/all_data.csv")
df.describe()


,FG_Id,CompanyId,RM_Id,SupplierId
count,2860.000000,2860.000000,2860.000000,2860.000000
mean,75.682517,33.330420,587.827972,20.029720
std,43.914524,16.301063,224.581264,9.319416
min,1.000000,1.000000,150.000000,1.000000
25%,37.000000,17.000000,404.750000,11.000000
50%,75.000000,33.000000,593.000000,24.000000
75%,118.000000,44.000000,735.000000,27.000000
max,149.000000,62.000000,1046.000000,40.000000


In [19]:
sku_parts = df["RM_SKU"].str.split("-")

df["RM_Code"] = sku_parts.str[:2].str.join("-")  # e.g. "RM-C28"
df["RM_hash"] = sku_parts.str[-1]  # trailing hex hash
df["RM_ingredient"] = sku_parts.str[2:-1].str.join(" ")  # middle parts joined


In [23]:
df.to_csv("../data/all_data.csv", index=False)


In [38]:
df[df["RM_ingredient"] == "citric acid"]


,FG_Id,FG_SKU,FG_Type,CompanyId,CompanyName,RM_Id,RM_SKU,RM_Type,SupplierId,SupplierName,RM_Code,RM_hash,RM_ingredient
35,3,FG-iherb-71022,finished-good,56,Ultima Replenisher,867,RM-C56-citric-acid-d55c874f,raw-material,27,Prinova USA,RM-C56,d55c874f,citric acid
36,3,FG-iherb-71022,finished-good,56,Ultima Replenisher,867,RM-C56-citric-acid-d55c874f,raw-material,38,Univar Solutions,RM-C56,d55c874f,citric acid
114,7,FG-iherb-105065,finished-good,27,Liquid I.V.,488,RM-C27-citric-acid-22adab04,raw-material,27,Prinova USA,RM-C27,22adab04,citric acid
115,7,FG-iherb-105065,finished-good,27,Liquid I.V.,488,RM-C27-citric-acid-22adab04,raw-material,38,Univar Solutions,RM-C27,22adab04,citric acid
372,23,FG-thrive-market-ultima-replenisher-passionfruit,finished-good,56,Ultima Replenisher,867,RM-C56-citric-acid-d55c874f,raw-material,27,Prinova USA,RM-C56,d55c874f,citric acid
373,23,FG-thrive-market-ultima-replenisher-passionfruit,finished-good,56,Ultima Replenisher,867,RM-C56-citric-acid-d55c874f,raw-material,38,Univar Solutions,RM-C56,d55c874f,citric acid
390,24,FG-thrive-market-drink-lmnt-electrolyte-drink-...,finished-good,26,LMNT,481,RM-C26-citric-acid-fa037e6f,raw-material,27,Prinova USA,RM-C26,fa037e6f,citric acid
391,24,FG-thrive-market-drink-lmnt-electrolyte-drink-...,finished-good,26,LMNT,481,RM-C26-citric-acid-fa037e6f,raw-material,38,Univar Solutions,RM-C26,fa037e6f,citric acid
582,33,FG-walmart-17018124,finished-good,41,Pedialyte,715,RM-C41-citric-acid-1ace1f97,raw-material,27,Prinova USA,RM-C41,1ace1f97,citric acid
583,33,FG-walmart-17018124,finished-good,41,Pedialyte,715,RM-C41-citric-acid-1ace1f97,raw-material,38,Univar Solutions,RM-C41,1ace1f97,citric acid


In [22]:
# import sys, importlib

# sys.path.insert(0, "..")
# import data.enrich_ingredients as ei

# importlib.reload(ei)

# df = ei.enrich("../data/all_data.csv")

# # Preview enriched columns
# df[["RM_ingredient", "aliases", "product_category", "region"]].drop_duplicates(
#     "RM_ingredient"
# ).head(20)
